# context_fusion.ipynb

TCFN: context fusion transformer + next-embedding predictor (negative result).

Paths are relative: `data/` holds dialogues and embeddings (not included), `models/` holds fine-tuned weights (not included). See the README.


## Setup: models

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

llm_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(llm_name)

local_embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)

victim_embedder = SentenceTransformer("intfloat/e5-base-v2", device=device)

llm.to(device)
llm.eval()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

## Context-fusion Transformer and training code

In [ ]:
class ContextFusionTransformer(nn.Module):
    def __init__(self, d_local, n_heads=8, n_layers=2, max_turns=25):
        super().__init__()
        self.pos_emb = nn.Embedding(max_turns, d_local)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_local,
            nhead=n_heads,
            dim_feedforward=d_local * 4,
            dropout=0.1,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_local, d_local)
        self.max_turns = max_turns

    def forward(self, turn_embs):
        """
        turn_embs: [n_turns, d_local]
        return:    [d_local]
        """
        n = turn_embs.size(0)
        positions = torch.arange(n, device=turn_embs.device)
        x = turn_embs + self.pos_emb(positions)
        x = x.unsqueeze(0)
        x = self.transformer(x)
        context = self.output_proj(x[0, -1, :])
        return context

class NextEmbeddingPredictor(nn.Module):
    def __init__(self, d, n_heads=8, n_layers=2):
        super().__init__()
        self.cls = nn.Parameter(torch.randn(1, 1, d))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=n_heads,
            dim_feedforward=4*d,
            batch_first=True,
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)

        self.pred_head = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Linear(d, d)
        )

    def forward(self, x):
        B = 1
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)

        x = self.transformer(x)

        context = x[:, 0, :]
        pred = self.pred_head(context)

        return pred

In [ ]:
class MultiTurnDataset(Dataset):
    def __init__(self, encoded_dialogs):
        """
        encoded_dialogs: list[tensor]
            Each element holds the turn embeddings of one dialogue, shape [n_turns, d_local]
            Only dialogues with n_turns >= 2 are kept
        """
        self.samples = []
        for dial_embs in encoded_dialogs:
            n = dial_embs.size(0)

            for t in range(1, n):
                history = dial_embs[:t]
                target  = dial_embs[t]
                self.samples.append((history, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_fn(batch):
    """
    Variable-length sequences are not padded; a list is returned
    Histories differ in length, so the DataLoader processes them one by one
    """
    histories = [item[0] for item in batch]
    targets   = torch.stack([item[1] for item in batch])
    return histories, targets

In [ ]:
def train_context_fusion(
    context_fusion,
    encoded_dialogs,
    n_epochs=30,
    batch_size=64,
    lr=1e-4,
    weight_decay=1e-2,
    val_ratio=0.1,
    save_path=None
):

    n_val = max(1, int(len(encoded_dialogs) * val_ratio))
    train_dialogs = encoded_dialogs[n_val:]
    val_dialogs   = encoded_dialogs[:n_val]

    train_dataset = MultiTurnDataset(train_dialogs)
    val_dataset   = MultiTurnDataset(val_dialogs)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )

    optimizer = torch.optim.AdamW(
        context_fusion.parameters(), lr=lr, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs
    )

    best_val_sim = -1.0

    for epoch in range(n_epochs):

        context_fusion.train()
        train_loss = 0.0

        for histories, targets in train_loader:
            targets = targets.to(device)

            preds = []
            for hist in histories:
                hist = hist.to(device)
                pred = context_fusion(hist)
                preds.append(pred)
            preds = torch.stack(preds)

            cos_loss = (1 - F.cosine_similarity(preds, targets)).mean()
            mse_loss = F.mse_loss(preds, targets)
            loss = cos_loss + 0.5 * mse_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(context_fusion.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()

        scheduler.step()

        context_fusion.eval()
        val_sims = []

        with torch.no_grad():
            for histories, targets in val_loader:
                targets = targets.to(device)
                preds = []
                for hist in histories:
                    hist = hist.to(device)
                    pred = context_fusion(hist)
                    preds.append(pred)
                preds = torch.stack(preds)
                sims = F.cosine_similarity(preds, targets)
                val_sims.extend(sims.cpu().tolist())

        val_sim = sum(val_sims) / len(val_sims)
        avg_train_loss = train_loss / len(train_loader)

        print(f"Epoch {epoch+1:02d}/{n_epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Cos Sim: {val_sim:.4f}")

        if val_sim > best_val_sim:
            best_val_sim = val_sim
            torch.save(context_fusion.state_dict(), save_path)
            print(f"  -> saved best model (val_sim={val_sim:.4f})")

    print(f"\nTraining finished, best val cosine similarity: {best_val_sim:.4f}")
    print(f"Model saved to {save_path}")

In [ ]:
def prepare_encoded_dialogs(dataset):
    encoded_dialogs = []
    for item in dataset:
      embed = []
      for d in item['dialogue']:
        embed.append(torch.tensor(d['embedding']).to(device))
      embed = torch.stack(embed, dim=0)
      encoded_dialogs.append(embed)
    return encoded_dialogs

## Training

In [ ]:
d_local = 768

context_fusion = ContextFusionTransformer(
  d_local=d_local,
  n_heads=8,
  n_layers=2,
  max_turns=25,
).to(device)

import json
with open("data/ill_embed.json", "r") as f:
  ill_data = json.load(f)
with open("data/bank_embed.json", "r") as f:
  bank_data = json.load(f)
with open("data/new_user_profile_embed.json", "r") as f:
  user_data = json.load(f)

encoded_dialog_ill = prepare_encoded_dialogs(ill_data[:79])
encoded_dialog_bank = prepare_encoded_dialogs(ill_data[:223])
encoded_dialog_user = prepare_encoded_dialogs(ill_data[:61])
print(f"Encoding Done")
encoded_dialogs = encoded_dialog_ill + encoded_dialog_bank + encoded_dialog_user

train_context_fusion(
  context_fusion=context_fusion,
  encoded_dialogs=encoded_dialogs,
  n_epochs=100,
  batch_size=64,
  lr=1e-4,
  save_path="models/new_context_fusion.pt",
)

## Recovery

In [ ]:
for p in local_embedder.parameters():
    p.requires_grad_(False)

In [ ]:
def encode_local(texts):
    return local_embedder.encode(
        texts,
        convert_to_tensor=True,
        device=device,
        show_progress_bar=False,
    )

def encode_victim(texts):
    return victim_embedder.encode(
        texts,
        convert_to_tensor=True,
        device=device,
        show_progress_bar=False,
    )

def format_context(prev_turns):
    """
    Concatenate the previously recovered turns in dialogue format
    Adjust to the chat template of the LLM in use
    """
    formatted = ""
    for i, turn in enumerate(prev_turns):
        if i % 2 == 0:
            formatted += f"User: {turn}"
        else:
            formatted += f"Assistant: {turn}"
    return formatted

def diversity_filter(token_embs, KS, T_hw):
    top_k = token_embs.size(0)
    selected_indices = []
    mask = torch.ones(top_k, dtype=torch.bool, device=token_embs.device)

    for i in range(top_k):
        if not mask[i]:
            continue
        selected_indices.append(i)
        if len(selected_indices) >= KS:
            break
        sims = F.cosine_similarity(token_embs[i].unsqueeze(0), token_embs, dim=1)
        sims[i] = 0
        mask = mask & (sims <= T_hw)
    return selected_indices

In [ ]:
def build_context_emb_transformer(prev_turns, window_llm=5):
    """
    prev_turns: list[str], previously recovered turns
    window_llm: only the most recent turns go into the LLM prefix, to bound the input length

    Returns:
        context_emb: [1, d_local], history embedding encoded by the Transformer
        llm_prefix: str, prefix text given to the LLM
    """
    if not prev_turns:
        return None, " "

    with torch.no_grad():
        turn_embs = encode_victim(prev_turns)
        context_emb = context_fusion(turn_embs).unsqueeze(0)

    recent_turns = prev_turns[-window_llm:]
    llm_prefix = format_context(recent_turns)

    return context_emb, llm_prefix

In [ ]:
import string

allowed_chars = set(string.ascii_letters + string.digits + " .,!?'\"-")

allowed_token_ids = []

for token, idx in tokenizer.get_vocab().items():

    text = tokenizer.decode([idx]).strip()

    if all(c in allowed_chars for c in text):
        allowed_token_ids.append(idx)

allowed_token_ids = set(allowed_token_ids)
vocab_size = len(tokenizer)
vocab_size = tokenizer.vocab_size

In [ ]:
def normalize(x):
    return (x - x.mean()) / (x.std() + 1e-8)

def compute_score(logprob, cos_sim, conf):
    return normalize(logprob) + conf * normalize(cos_sim)

def update_W(E, E_tilde):
    d = E.shape[1]
    I = torch.eye(d, device=device, dtype=torch.float32)
    return torch.linalg.inv(E.T @ E + 0.1 * I) @ E.T @ E_tilde

In [ ]:
gpu_total_mem = torch.cuda.get_device_properties(0).total_memory
threshold = 0.8 * gpu_total_mem

In [ ]:
def generate_text(max_len, target_embedding, top_k=100, KS=50, gamma=0.8, alpha=1, gamma_hist=0.9, T_hw=0.9):
    beam = [" "]
    W_prev = None
    V_prev = None
    E_tilde_prev = None
    special_index = 1
    flag = False
    beam_size = 5

    for t in range(max_len):
        inputs = tokenizer(beam, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            outputs = llm(**inputs)
            lengths = inputs['attention_mask'].sum(dim=1)
            logits = outputs.logits[torch.arange(len(beam)), lengths-1, :]
            probs = torch.softmax(logits, dim=-1)
            topk_probs, topk_ids = torch.topk(probs, top_k, dim=-1)
        embedding_layer = llm.get_input_embeddings()
        token_embs = embedding_layer(topk_ids)
        candidates = []
        logprobs = []
        for b_idx, text in enumerate(beam):
            selected_indices = diversity_filter(token_embs[b_idx], KS, T_hw)
            for idx in selected_indices:
                token_id = topk_ids[b_idx, idx].item()
                logprobs.append(torch.log(topk_probs[b_idx, idx]))
                candidates.append(text + tokenizer.decode([token_id], skip_special_tokens=True))

        logprobs = torch.stack(logprobs)
        del inputs, outputs, logits, probs, token_embs, topk_probs, topk_ids

        E = torch.tensor(local_embedder.encode(candidates), dtype=torch.float32, device=device)

        if W_prev is None:
            num_queries = int(KS * (gamma ** (t)))
            top50_idx = torch.topk(logprobs, num_queries).indices
            best_texts = [candidates[i] for i in top50_idx.tolist()]
            E_tilde = torch.tensor(
                victim_embedder.encode(best_texts),
                dtype=torch.float32,
                device=device
            )

            W = update_W(E[top50_idx], E_tilde)

            E_proj = E @ W
            cos_sims = F.cosine_similarity(E_tilde, target_embedding)

            E_proj_top50 = E[top50_idx] @ W
            cos_sims_tilde = F.cosine_similarity(E_proj_top50, E_tilde, dim=1)
            conf1 = 0.7 * cos_sims_tilde.mean()

            logprobs_50 = logprobs[top50_idx]
            scores = compute_score(logprobs_50, cos_sims, conf1)
            final_idx = torch.topk(scores, beam_size).indices.tolist()
            beam = [candidates[i] for i in final_idx]

            V_prev = top50_idx.tolist()
            W_prev = W
            W_prev_another = W
            E_tilde_prev = E_tilde
            special_index += 1

            del E, E_proj, cos_sims, scores
            continue

        E_proj = E @ W_prev

        cos_sims = F.cosine_similarity(E_proj, target_embedding)
        sims = []
        for idx, i in enumerate(V_prev):
            sims.append(
                F.cosine_similarity(
                    (E[i] @ W_prev).unsqueeze(0),
                    E_tilde_prev[idx].unsqueeze(0)
                )
            )
        conf_t = torch.mean(torch.stack(sims))
        scores_all = compute_score(logprobs, cos_sims, conf_t)

        del cos_sims, E_proj

        sorted_idx = torch.argsort(scores_all)
        mid = len(sorted_idx) // 2

        num_queries = int(KS * (gamma ** (t)))
        best_group = sorted_idx[num_queries:]
        worst_group = sorted_idx[:num_queries]

        top50_group = best_group
        best_texts = [candidates[i] for i in top50_group.tolist()]
        E_tilde = victim_embedder.encode(best_texts, convert_to_tensor=True, device=device)
        E_best = E[top50_group]
        W = update_W(E_best, E_tilde)

        E_worst = E[worst_group]
        E_worst_proj = E_worst @ W
        combined_E = torch.cat([E_tilde, E_worst_proj], dim=0)

        combined_cos = F.cosine_similarity(combined_E, target_embedding)
        combined_logprob = torch.cat([logprobs[top50_group], logprobs[worst_group]])

        cos_sims_tilde1 = F.cosine_similarity(E_best @ W_prev_another, E_tilde, dim=1)
        conf2 = cos_sims_tilde1.mean()

        scores_final = compute_score(combined_logprob, combined_cos, conf2)
        combined_texts = best_texts + [candidates[i] for i in worst_group.tolist()]

        final_idx = torch.topk(scores_final, beam_size).indices.tolist()
        beam = [combined_texts[i] for i in final_idx]

        V_prev = top50_group.tolist()
        if special_index > 2:
          W_prev = W_prev_another
          W_prev_another = W
        else:
          W_prev_another = W
          special_index += 1
        E_tilde_prev = E_tilde

        del E, E_tilde, W, scores_all
        del E_best, E_worst, E_worst_proj
        del combined_E, combined_cos, combined_logprob, scores_final
        mem_used = torch.cuda.memory_allocated()
        if mem_used > threshold:
          print(f"Step {t}: memory {mem_used/1e9:.2f} GB exceeds threshold, clearing cache")
          torch.cuda.empty_cache()

    return beam

In [ ]:
import string

allowed_chars = set(string.ascii_letters + string.digits + " .,!?'\"-")

In [ ]:
def new_train_step_multiturn(target_embs, max_len, prev_turns,
                         alpha=1.0, beta=0.05, gamma=0.05):

    batch_size = len(target_embs)
    topk = 100
    KS = 50
    T_hw = 0.9
    beam_width = 5

    context_emb, llm_prefix = build_context_emb_transformer(prev_turns)

    beams = [[(llm_prefix, 0.0)] for _ in range(batch_size)]

    for t in range(max_len):
        batch_prefix_idx = []
        batch_prefix = []
        batch_prefix_score = []

        for b_idx, beam_list in enumerate(beams):
            for prefix, score in beam_list:
                batch_prefix_idx.append(b_idx)
                batch_prefix.append(prefix)
                batch_prefix_score.append(score)

        if not batch_prefix:
            break

        batch_input_ids = tokenizer(
            batch_prefix, return_tensors="pt", padding=True
        ).input_ids.to(device)

        with torch.no_grad():
            batch_logits = llm(batch_input_ids).logits[:, -1, :]
        batch_log_probs = F.log_softmax(batch_logits, dim=-1)

        all_new_texts = []
        all_candidate_meta = []

        for i, b_idx in enumerate(batch_prefix_idx):

            prefix = batch_prefix[i]
            prev_score = batch_prefix_score[i]

            top_ids = batch_log_probs[i].topk(topk, dim=-1).indices

            with torch.no_grad():
                token_embs = llm.get_input_embeddings()(top_ids)

            diverse_indices = diversity_filter(token_embs, KS=KS, T_hw=T_hw)

            for idx in diverse_indices:

                token_id = top_ids[idx].item()
                lang_score = batch_log_probs[i][token_id]

                token_str = tokenizer.decode(token_id, skip_special_tokens=True)

                new_text = prefix + (" " if not token_str.startswith(" ") else "") + token_str

                all_new_texts.append(new_text)

                all_candidate_meta.append((b_idx, new_text, lang_score, prev_score))

        if not all_new_texts:
            break

        with torch.no_grad():
            all_local_embs = encode_local(all_new_texts)
            temp = []
            for embed in all_local_embs:
                single_pred_v = context_fusion(embed.unsqueeze(0))
                temp.append(single_pred_v)

        candidate_scores = []

        for k, (b_idx, new_text, lang_score, prev_score) in enumerate(all_candidate_meta):

            pred_v = temp[k]
            target_v = target_embs[b_idx]

            pred_score = F.cosine_similarity(pred_v, target_v, dim=-1)

            if context_emb is not None:
                curr_emb = temp[k]
                context_score = F.cosine_similarity(curr_emb, context_emb, dim=-1)
            else:
                context_score = 0.0

            final_score = (
                0 +
                alpha * normalize(pred_score) +
                beta  * lang_score +
                gamma * context_score
            )

            candidate_scores.append((final_score, b_idx, new_text))

        from collections import defaultdict
        grouped = defaultdict(list)

        for score, b_idx, new_text in candidate_scores:
            grouped[b_idx].append((score, new_text))

        new_beams = [[] for _ in range(batch_size)]

        for b_idx in range(batch_size):

            top_candidates = sorted(
                grouped[b_idx],
                key=lambda x: x[0],
                reverse=True
            )[:beam_width]

            new_beams[b_idx] = [
                (text, score) for score, text in top_candidates
            ]

        beams = new_beams

        print([[t for t, _ in b] for b in beams])

    results = []

    for b_idx in range(batch_size):
        best = beams[b_idx][0][0] if beams[b_idx] else ""
        recovered = best[len(llm_prefix):].strip()
        results.append(recovered)

    return results

In [ ]:
context_fusion = ContextFusionTransformer(
  d_local=768,
  n_heads=8,
  n_layers=2,
  max_turns=25,
).to(device)
context_fusion.load_state_dict(torch.load("models/new_context_fusion.pt"))
context_fusion.eval()
for p in context_fusion.parameters():
    p.requires_grad_(False)

In [ ]:
import json
with open("data/ill_embed.json", "r") as f:
  ill_data = json.load(f)
with open("data/bank_embed.json", "r") as f:
  bank_data = json.load(f)
with open("data/new_user_profile_embed.json", "r") as f:
  user_data = json.load(f)

In [ ]:
def recover_multiturn(all_leaked_embs, max_len):
    recovered_turns = []

    for turn_idx, leaked_emb in enumerate(all_leaked_embs):
        print(f"Recovering turn {turn_idx + 1} / {len(all_leaked_embs)}...")
        if turn_idx == 0:
          text = generate_text(max_len, leaked_emb)
          recovered_turns.append(text[0])
          print(f"Recovered turn {turn_idx + 1}: {text[0]}\n")

        result = new_train_step_multiturn(
            target_embs=[leaked_emb.unsqueeze(0)],
            max_len=max_len,
            prev_turns=recovered_turns,
        )

        recovered_turns.append(result[0])
        print(f"Recovered turn {turn_idx + 1}: {result[0]}\n")

    return recovered_turns

## Example

In [ ]:
# Qualitative example: reconstruct one medical dialogue
for item in ill_data[38:39]:
  embed = []
  max_l = []
  for d in item['dialogue']:
    text = d['text']
    length = int(len(text.split()) * 1.2)
    max_l.append(length)
    embed.append(torch.tensor(d['embedding']).to(device))
  recover_multiturn(embed, max(max_l))